# Lab 1 — From Raw Data to a First Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/INCORTX/INCORTX.github.io/blob/master/DataAnalytics/session-01/lab_01.ipynb)

**Data Analytics · DTii 1/2026 · Lab session 1** · about 180 minutes

### By the end of this session you will be able to
1. **See where a dataset is dirty** and decide what to do about it, with a reason you can defend
2. **Write a `Pipeline` with no data leakage** — the most common mistake in this work, and the one you cannot spot by eye
3. **Judge whether a model is actually any good**, by comparing it to a baseline and choosing a metric that fits the problem

### How to use this notebook

| Mark | Meaning |
|---|---|
| 🔵 **Guided** | Run along with the instructor · read the text before you press Run |
| 🟠 **Your-Turn** | You write it · look for the `# TODO` cell |
| ✅ **Expected** | What you should get, so you can check yourself |
| 💥 **This cell is meant to fail** | Read the error until you understand it, then move on |

> **Every dataset here loads in one line.** No file uploads, no Google Drive, no paths to fix.
> Chart text is in English so the figures render the same on every machine.

In [ ]:
# ── Setup: run this cell first ──────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

SEED = 42                      # every random step in this notebook uses this
np.random.seed(SEED)

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 30)
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('pandas      ', pd.__version__)
print('seaborn     ', sns.__version__)
import sklearn; print('scikit-learn', sklearn.__version__)

---
# A · A Model in 15 Minutes — Then the Real World Arrives
**about 30 minutes**

We build a working model **first**, and only then look at what breaks it. The order matters:
learn data cleaning before you have seen a model fail, and cleaning feels like chores nobody explained.

### 🔵 A.1 — The dataset we use all session

`titanic` holds 891 passenger records. The goal is to predict who **survived** (`survived` = 1) and who did not (`survived` = 0).

We chose it because it is **genuinely messy on its own** — none of the mess below was manufactured for teaching.

In [ ]:
df = sns.load_dataset('titanic')      # one line, no upload, no Drive
print('shape:', df.shape)
df.head()

### 🔵 A.2 — First model, three columns only

Start with three numeric columns that have **no missing values**, so the code runs at all:

| Column | Meaning |
|---|---|
| `pclass` | Ticket class: 1 / 2 / 3 |
| `sibsp` | Siblings and spouses aboard |
| `fare` | Ticket price |

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X3 = df[['pclass', 'sibsp', 'fare']]
y  = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X3, y, test_size=0.2, random_state=SEED, stratify=y)

tree = DecisionTreeClassifier(random_state=SEED)
tree.fit(X_train, y_train)

acc = accuracy_score(y_test, tree.predict(X_test))
print(f'Accuracy: {acc:.4f}')

✅ **Expected:** `Accuracy: 0.6480`

Six lines and the model is right about 65% of the time. That sounds usable.

**But is it?** You cannot answer that yet — not until you know what *guessing without a model at all* would score.

### 🔵 A.3 — The line you have to beat (baseline)

Most passengers did not survive. So the dumbest possible model is: **say "did not survive" for everyone**, without looking at any data.

`DummyClassifier` does exactly that, and the number it produces is the line a real model has to clear.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)

base_acc = accuracy_score(y_test, baseline.predict(X_test))
print(f'Baseline accuracy: {base_acc:.4f}   <- guessing "did not survive" every time')
print(f'Our tree         : {acc:.4f}')
print(f'Improvement      : {acc - base_acc:+.4f}')

✅ **Expected:** baseline `0.6145` · tree `0.6480` · improvement `+0.0335`

**This is the most expensive lesson in the session.** That respectable-looking 65% is worth exactly
**3.4 points** more than guessing.

> **Rule to keep:** a model's number means nothing without something to compare it against.
> Every time you report a result, the baseline goes next to it.

### 🔵 A.4 — What did the model actually learn?

A decision tree can be **read as sentences**, which most other models cannot.

In [ ]:
from sklearn.tree import plot_tree

small_tree = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_train, y_train)
print(f'depth-3 accuracy: {accuracy_score(y_test, small_tree.predict(X_test)):.4f}')

plt.figure(figsize=(15, 6))
plot_tree(small_tree, feature_names=X3.columns, class_names=['died', 'survived'],
          filled=True, rounded=True, fontsize=9, impurity=False)
plt.title('What the tree learned (first 3 levels)')
plt.tight_layout(); plt.show()

✅ **Expected:** `depth-3 accuracy: 0.6425`, plus a tree you can read

Try reading the leftmost branch out loud, something like
*"if the fare is below X and the passenger is in third class, predict died."*

**Notice:** the three-level tree scores 0.6425 and the unlimited one scores 0.6480 — nearly the same,
despite being far more complex. Hold on to that observation; we come back to it in part D.

### 💥 A.5 — Then the real world arrives

We hand-picked three clean columns. On a real job nobody picks them for you.

**The next cell is meant to fail.** Throw the whole DataFrame at the same model and read the error.

In [ ]:
X_all = df.drop(columns=['survived'])

try:
    DecisionTreeClassifier(random_state=SEED).fit(X_all, y)
except ValueError as e:
    print('ValueError:', e)

✅ **Expected:** `ValueError: could not convert string to float: 'male'`

scikit-learn only takes numbers, so a column like `sex` holding `'male'` / `'female'` stops it dead.

That is **problem one**, and we are lucky with it, because it makes a noise. The rest are quieter.

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].to_frame('missing_rows')
missing['percent'] = (missing['missing_rows'] / len(df) * 100).round(1)
print(missing)

✅ **Expected:**

| Column | Missing | % |
|---|---:|---:|
| `age` | 177 | 19.9 |
| `embarked` | 2 | 0.2 |
| `deck` | 688 | 77.2 |
| `embark_town` | 2 | 0.2 |

**Problem two:** `deck` is missing for **77%** of passengers. Filling it in is wrong, throwing it away wastes
what is there. There is no textbook answer — decisions like this are the actual daily work of a data analyst.

### 🟠 Your-Turn A — Find out how many ways this data is broken

Explore with `df.info()` · `df.describe()` · `df.nunique()` · `df.duplicated().sum()`,
then **write three problems** in the markdown cell below.

Each one needs **(a)** what the problem is, **(b)** which column, **(c)** what goes wrong if you ignore it.

✅ **Expected:** at least three of — text columns a model cannot take · missing values in `age`/`deck` ·
repeated rows · columns that say the same thing twice · fares that jump to extremes

In [ ]:
# TODO: explore with the commands above, then write your three problems in the next cell

**Your group's answer (write here):**

1.
2.
3.

---
# B · Data Collection — Getting the Data In
**about 15 minutes**

Your group project starts from nothing, not from a DataFrame somebody prepared. If you cannot load data,
you do not have a project.

Every one of the twenty assignment topics loads from a direct link or a single library call, so this
is short: read a file, and know the three things that go wrong when you do.

### 🔵 B.1 — A CSV that lives on the web

`read_csv` takes a URL directly. No downloading first.

In [ ]:
CSV_URL = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv'

penguins = pd.read_csv(CSV_URL)
print('shape:', penguins.shape)
penguins.head(3)

✅ **Expected:** `shape: (344, 7)`

pandas reads Excel and JSON the same way — `pd.read_excel(url)` · `pd.read_json(url)`.

### 🔵 B.2 — Three traps that show up in every real file

Files from Thai government portals hit all three of these regularly.

In [ ]:
from io import StringIO

raw = '''date,province,population,revenue
2026-01-15,กรุงเทพมหานคร,"5,494,932","1,240,500.75"
2026-02-15,เชียงใหม่,"1,798,120","320,900.00"'''

bad = pd.read_csv(StringIO(raw))
print('-- read naively --')
print(bad.dtypes)

good = pd.read_csv(StringIO(raw), thousands=',', parse_dates=['date'])
print('\n-- read with thousands= and parse_dates= --')
print(good.dtypes)
print()
print(good)

✅ **Expected:** the first read gives `population` and `revenue` as `object` (text) — you cannot do arithmetic on them.
The second gives `int64` / `float64`, and `date` as `datetime64`.

**The three traps:**
1. **Thousands separators.** `"5,494,932"` reads as text · fix with `thousands=','`
2. **Dates** read as text · fix with `parse_dates=[...]`
3. **Thai encoding.** Files from older systems are often `cp874`, not `utf-8`.
   If the characters come out garbled, try `pd.read_csv(path, encoding='cp874')`

### 📋 Assignment 1 — pick one of these 20

**Groups of four. One topic per group, first come first served, no two groups on the same one.**
You keep this dataset for the rest of the course — it is the one you will walk end to end in session 5.

Take the topic you pick and go the whole way:

```
real data -> EDA -> preprocessing (Pipeline) -> baseline -> model -> a metric that fits -> limitations
```

📄 **[How it is marked, how the questions work, what to hand in](https://classes.incortx.com/DataAnalytics/session-00/)** — read that before you start.
The short version: you need **a baseline to beat**, a **`Pipeline` with no leakage**, and **a metric that
fits the problem**. A model that loses to its own baseline still scores full marks if you explain why.

**The 20 topics.** Difficulty: 🟢 the data is ready to use · 🟡 it needs real cleaning or merging first. Every dataset here already exists and is free to download — none of them ask you to collect or scrape data yourself, so a week is enough for any of them.

The **trap** column is not a general warning. It is the specific thing that will bite you in that dataset,
and it is where the code questions will come from.

**Society, prices & environment**

| # | Topic — the question to answer | Data | The trap you will hit | |
|:--:|---|---|---|:--:|
| **1** | Who earns above the census threshold? | UCI Adult / Census Income — `archive.ics.uci.edu/static/public/2/adult.zip` (ใช้ `adult.data` · ไม่มีหัวตาราง ต้องตั้งชื่อคอลัมน์เอง) | `fnlwgt` is a survey sampling weight, not a fact about the person. It describes how the census was drawn, so feeding it to the model leaks the study design and means nothing at prediction time. The `sex` and `race` columns also make this the one topic where you should say what you chose not to use. | 🟢 |
| **2** | What is this district of houses worth? | `sklearn.datasets.fetch_california_housing` — one call, no download page | The target is capped: 965 of the 20,640 districts (4.7%) sit at exactly 5.0 because the census clipped the value there. Your model can never be right about those, and no amount of tuning fixes it — you have to notice the ceiling and say so. | 🟡 |
| **3** | What should this diamond cost? | `seaborn.load_dataset('diamonds')` — one line | `cut`, `color` and `clarity` are ordered qualities, but the order is not alphabetical. Sorting the clarity labels puts `IF` — the best grade — second, right after the worst. Encode them alphabetically and you teach the model an order that is simply wrong. | 🟢 |
| **4** | What will the pollution reading be next hour? | UCI Air Quality — `archive.ics.uci.edu/static/public/360/air+quality.zip` | Missing readings are written as **-200**, not left blank: 16,701 of them across 13 columns, and `isnull()` reports none of it. It is also a time series, so the split has to follow the clock. | 🟡 |
| **5** | How much forest will this fire burn? | UCI Forest Fires — `archive.ics.uci.edu/static/public/162/forest+fires.zip` | Almost half the fires (247 of 517) burned exactly 0.00 ha, while the largest burned 1090.8. Average error is dominated by the pile of zeros, so a model that always predicts zero looks respectable. Decide what you are actually predicting before you pick a metric. | 🟡 |

**Business & finance**

| # | Topic — the question to answer | Data | The trap you will hit | |
|:--:|---|---|---|:--:|
| **6** | Which customers are about to leave? | `fetch_openml('Telco-Customer-Churn', version=1)` — one call, no account | Churn runs at 26.5% against 73.5% staying, and you have to decide whether recall or precision matters more to the team who will phone these customers. | 🟢 |
| **7** | Which cardholders will default next month? | UCI Default of Credit Card Clients — `archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip` (เป็นไฟล์ .xls ต้อง `pip install xlrd` และอ่านด้วย `header=1`) | A false negative and a false positive cost very different amounts. Several categorical columns are stored as integers and only some of them are genuinely ordinal. | 🟢 |
| **8** | Which transactions are fraud? | `fetch_openml('creditcard', version=1)` — one call, no account | Only 0.17% of rows are fraud. Answering "not fraud" every single time scores 99.8% accuracy, so the whole job is proving you beat that. | 🟡 |
| **9** | Who will accept the offer made by phone? | UCI Bank Marketing — `archive.ics.uci.edu/static/public/222/bank+marketing.zip` (ข้างในมี `bank.zip` ซ้อนอีกชั้น · ใช้ `bank-full.csv` คั่นด้วย `;`) | The `duration` column is the length of the call, which you only know once the call has ended. Keeping it gives a beautiful score and a useless model. | 🟢 |
| **10** | How many bikes will be hired this hour? | UCI Bike Sharing — `archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip` (ใช้ `hour.csv`) | `casual` and `registered` add up to the target exactly. Leave either one in the features and you score almost perfectly while predicting nothing at all. | 🟢 |

**Health & science**

| # | Topic — the question to answer | Data | The trap you will hit | |
|:--:|---|---|---|:--:|
| **11** | Screen for diabetes risk from basic measurements | Pima Indians Diabetes — `raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv` (ไม่มีแถวหัวตาราง ต้องตั้งชื่อคอลัมน์เอง) | Zeros in glucose, BMI and blood pressure are missing values in disguise. `isnull()` will not find them — only `describe()` and some thought will. | 🟢 |
| **12** | Which heart-failure patients are most at risk? | UCI Heart Failure Clinical Records — `archive.ics.uci.edu/static/public/519/heart+failure+clinical+records.zip` | Only 299 rows. A single train/test split proves nothing here — you need cross-validation and you must report the spread. | 🟢 |
| **13** | Predict wine quality from its chemistry | UCI Wine Quality — `archive.ics.uci.edu/static/public/186/wine+quality.zip` (มีทั้งไวน์แดงและขาว คั่นด้วย `;`) | The target is an ordinal score from 3 to 9. You have to choose classification or regression yourself, and defend the choice. | 🟢 |
| **14** | Is this mushroom safe to eat? | UCI Mushroom — `archive.ics.uci.edu/static/public/73/mushroom.zip` (ใช้ `agaricus-lepiota.data` · ไม่มีหัวตาราง · คลาสอยู่คอลัมน์แรก) | You will hit 100% accuracy almost immediately. The real assignment is explaining why this problem is that easy, and saying when a result that good is a warning. | 🟢 |
| **15** | Is this tumour benign or malignant? | `sklearn.datasets.load_breast_cancer` — ships with the library, no download | The 30 columns are 10 measurements seen three ways (mean, error, worst), so they are heavily correlated and the importance of a real signal gets split three ways, making it look weak. | 🟢 |

**Text & behaviour**

| # | Topic — the question to answer | Data | The trap you will hit | |
|:--:|---|---|---|:--:|
| **16** | How many stars did this Thai review give? | Wongnai review corpus, 40,000 reviews — `raw.githubusercontent.com/wongnai/wongnai-corpus/master/review/review_dataset.zip` (semicolon-separated, no header row) | Thai has no spaces between words, so you must segment before anything else, and negation flips a whole sentence in a way bag-of-words cannot see. The ratings are lopsided too: 4 and 5 stars are 64% of the reviews while 1 star is 1%, so answering "4" every time already scores 46.9%. | 🟡 |
| **17** | Is this message spam? | UCI SMS Spam Collection — `archive.ics.uci.edu/static/public/228/sms+spam+collection.zip` (ไฟล์ `SMSSpamCollection` คั่นด้วย tab ไม่มีหัวตาราง) | A false positive — a real message dropped in the spam bin — costs far more than a false negative. You must pick your own threshold instead of accepting 0.5. | 🟢 |
| **18** | Which students are at risk of failing? | UCI Student Performance — `archive.ics.uci.edu/static/public/320/student+performance.zip` (มี `student.zip` ซ้อนอีกชั้น · ใช้ `student-mat.csv` คั่นด้วย `;`) | G1 and G2 (earlier grades) predict G3 almost perfectly. Whether to keep or drop them is a real decision, and the answer differs for "predict before term starts" versus "warn mid-term". | 🟢 |
| **19** | Will this viewer rate the film highly? | MovieLens — `files.grouplens.org/datasets/movielens/ml-latest-small.zip` (ใช้ `ratings.csv` + `movies.csv`) | A random split by row puts the same viewer in both train and test, so the model learns individual people rather than taste. You have to split by user. | 🟡 |
| **20** | Which newsgroup did this post come from? | `sklearn.datasets.fetch_20newsgroups` — one call | The raw posts keep their email headers, and the headers name the group. Accuracy runs 0.986 with them and 0.918 once you pass `remove=("headers","footers","quotes")` — the first number is the metadata answering for you, not the text. | 🟢 |

In [ ]:
# ── Record your group's choice ─────────────────────────────────────────────
TOPIC_ID = None      # <- put your group's topic number here, then run this cell

TOPIC_TRAPS = {
     1: ('Who earns above the census threshold?',
        '`fnlwgt` is a survey weight, not a property of the person'),
     2: ('What is this district of houses worth?',
        'target is clipped at 5.0 - 4.7% of rows sit on the ceiling'),
     3: ('What should this diamond cost?',
        'ordinal grades whose alphabetical order is wrong (IF sorts 2nd)'),
     4: ('What will the pollution reading be next hour?',
        '-200 means missing (16,701 of them) and isnull() sees nothing'),
     5: ('How much forest will this fire burn?',
        'half the targets are exactly zero - predicting 0 looks fine'),
     6: ('Which customers are about to leave?',
        'recall or precision? decide for the team making the calls'),
     7: ('Which cardholders will default next month?',
        'FN and FP cost different amounts; integer columns are not all ordinal'),
     8: ('Which transactions are fraud?',
        '0.17% positives - "never fraud" already scores 99.8%'),
     9: ('Who will accept the offer made by phone?',
        '`duration` is only known after the call - target leakage'),
    10: ('How many bikes will be hired this hour?',
        '`casual` + `registered` = the target, exactly'),
    11: ('Screen for diabetes risk from basic measurements',
        'zeros in glucose/BMI are missing values isnull() cannot see'),
    12: ('Which heart-failure patients are most at risk?',
        '299 rows - one split proves nothing, use cross-validation'),
    13: ('Predict wine quality from its chemistry',
        'ordinal target - you choose classification or regression, and defend it'),
    14: ('Is this mushroom safe to eat?',
        'you will get 100% - the work is explaining why'),
    15: ('Is this tumour benign or malignant?',
        '30 columns = 10 measurements x 3 views, so importances split'),
    16: ('How many stars did this Thai review give?',
        'segment Thai first; answering 4 every time already scores 46.9%'),
    17: ('Is this message spam?',
        'false positives cost more - choose your own threshold'),
    18: ('Which students are at risk of failing?',
        'G1/G2 nearly give away G3 - keep or drop depends on the use'),
    19: ('Will this viewer rate the film highly?',
        'split by user, not by row - the same person leaks across'),
    20: ('Which newsgroup did this post come from?',
        'headers name the answer: 0.986 with them, 0.918 without'),
}

if TOPIC_ID in TOPIC_TRAPS:
    title, trap = TOPIC_TRAPS[TOPIC_ID]
    print(f'Topic {TOPIC_ID}: {title}')
    print(f'Watch out for : {trap}')
else:
    print('Set TOPIC_ID to your group number (1-20) and run this cell again.')

✅ **Expected:** your topic and its trap printed back at you. Write the trap somewhere you will see it again —
it is the first thing you should check when your results look strange.

### 🟠 Your-Turn B — Load your own group's data

Pick your topic from the list above and get that dataset into a DataFrame.

Print three things: **the row and column count** · **`.head()`** · **the source and its licence**.

✅ **Expected:** a DataFrame with real data in it. If you get stuck, raise your hand —
do not move on to part C without loading something.

In [ ]:
# TODO: load your group's data
# GROUP_TOPIC = '...'
# SOURCE_URL  = '...'
# LICENSE     = '...'

---
# C · Data Preprocessing — Making the Data Usable
**about 50 minutes**

This is the longest part of the session because it is where the time actually goes in this job,
and where people go wrong without noticing.
**The most dangerous mistake is the one that makes your results look better.**

### 🔵 C.1 — Repeated rows, and the trap hiding in them

In [ ]:
print('rows flagged as duplicated:', df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values(['pclass', 'sex', 'age']).head(6)

✅ **Expected:** `rows flagged as duplicated: 107`

**Do not reach for `drop_duplicates()` yet.**

The Titanic carried 891 distinct passengers. `duplicated()` flags 107 rows because this dataset
**has no identifier column**, and plenty of third-class men happen to match on every single field.

> **Rule:** `duplicated()` tells you *the values repeat*. It does not tell you *it is the same person*.
> Delete here and you delete 107 real people. Before dropping duplicates, always ask
> **what makes one row different from another** in this dataset.

### 🔵 C.2 — Missing values: `deck` is 77% empty, now what?

There is no right answer, only a decision you can defend:

| Option | What you gain | What you lose |
|---|---|---|
| **Drop the column** | Nothing to guess at | The 203 passengers who do have a value |
| **Drop the rows** | Everything left is complete | **203 rows out of 891** — you threw away 77% of the data |
| **Fill with the most common value** | Every row survives | 688 rows get a value you invented = a fake signal |
| **Turn it into "cabin known / unknown"** | The missingness itself becomes data | Still a guess that missingness means something |

**What we will do:** drop `deck` — because the 77% is not missing at random.
Third-class passengers almost never had a cabin recorded, so filling it in means inventing a story
about the largest group in the data.

In [ ]:
print('deck present by class:')
print(df.groupby('pclass')['deck'].apply(lambda s: f'{s.notna().sum():3d} / {len(s):3d}'))

✅ **Expected:** class 1 has 175/216 · class 2 has 16/184 · class 3 has 12/491

That is the evidence that the missingness is not random — which changes the answer to
"should we fill it in?" completely.

### 💥 C.3 — The trap that will make you happy for the wrong reason

Now let us deal with the text columns. The fastest way is `pd.get_dummies()`, which turns
everything into numbers in one go.

**Run it and look hard at the number.**

In [ ]:
from sklearn.model_selection import train_test_split

naive = pd.get_dummies(df.drop(columns=['survived'])).fillna(0)

Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(
    naive, y, test_size=0.2, random_state=SEED, stratify=y)

naive_model = DecisionTreeClassifier(random_state=SEED).fit(Xn_tr, yn_tr)
print(f'Accuracy: {accuracy_score(yn_te, naive_model.predict(Xn_te)):.4f}')

✅ **Expected:** `Accuracy: 1.0000`

**Every single prediction correct.** If that feels good, stop and get suspicious instead.

No real problem is answered perfectly. **A result that is too good is almost always a sign something is wrong.**

In [ ]:
importance = pd.Series(naive_model.feature_importances_, index=naive.columns)
print(importance.sort_values(ascending=False).head(5).round(4))

✅ **Expected:** `alive_no  1.0`, and every other feature at `0.0`

The model learned nothing. It found **the answer sitting in the input.**

In [ ]:
print(pd.crosstab(df['survived'], df['alive']))

✅ **Expected:** a perfect diagonal — `survived=0` with `alive='no'` 549 times, `survived=1` with `alive='yes'` 342 times

The `alive` column **is the target, spelled as words.**

> ### 🎯 Target leakage
> Target leakage is when the input contains something you could only know **after** you know the answer.
> The model scores beautifully in testing and then **fails completely in production**, because that column
> is not there when you need it.
>
> Symptoms to suspect immediately: unusually high accuracy, or a single feature holding almost all the importance.

### 🔵 C.3.1 — Which columns to remove, and why each one

| Column | Reason |
|---|---|
| `alive` | **The answer itself** — full leakage |
| `class` | Identical to `pclass` in every row (First/Second/Third ↔ 1/2/3) |
| `who` · `adult_male` | Derived from `sex` + `age`, which we already have — no new information |
| `deck` | 77% missing, and not at random (see C.2) |

Redundant columns are not as harmful as leakage, but they make `feature_importances_` hard to read,
because the importance gets split between columns that say the same thing.

In [ ]:
LEAKY_OR_DUPLICATE = ['alive', 'class', 'who', 'adult_male', 'deck']

data = df.drop(columns=LEAKY_OR_DUPLICATE)
X = data.drop(columns=['survived'])
y = data['survived']

print('columns kept:', list(X.columns))

✅ **Expected:** `['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'embark_town', 'alone']`

### 🔵 C.4 — Outliers: an extreme value is not automatically a wrong value

In [ ]:
q1, q3 = X['fare'].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
outliers = X['fare'] > upper

print(f'IQR upper fence : {upper:.2f}')
print(f'rows above fence: {outliers.sum()}  (max fare {X["fare"].max():.2f})')
print(f'survival rate  -> outliers {y[outliers].mean():.3f} vs everyone {y.mean():.3f}')

plt.figure(figsize=(7, 2.5))
plt.boxplot(X['fare'], vert=False, widths=0.6)
plt.axvline(upper, color='tab:red', ls='--', lw=1)
plt.text(upper + 8, 1.28, f'IQR fence = {upper:.0f}', color='tab:red', fontsize=9)
plt.xlabel('Ticket fare')
plt.title('116 fares sit above the fence - and most of those passengers survived')
plt.yticks([]); plt.tight_layout(); plt.show()

✅ **Expected:** fence at `65.63` · `116` rows above it ·
**those passengers survived at `0.681` against `0.384` for everyone else**

Delete the outliers by the IQR formula and you delete **the strongest signal in the dataset** —
the people who paid the most were first-class passengers who reached the lifeboats first.

> **Rule:** outliers come in two kinds. *Recording errors* (an age of 999) should be fixed or removed.
> *Rare but real values* should be kept. Telling them apart takes domain knowledge, not a formula.

### 🔵 C.5 — Scaling: why trees do not care and kNN cannot live without it

kNN decides by **distance**. If one column is measured in far larger units than another,
it swamps the whole calculation.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

NUM = ['age', 'sibsp', 'parch', 'fare', 'pclass']
print(X[NUM].describe().loc[['min', 'max']].round(2))

Xk_tr, Xk_te, yk_tr, yk_te = train_test_split(
    X[NUM], y, test_size=0.2, random_state=SEED, stratify=y)

plain = Pipeline([('imp', SimpleImputer(strategy='median')),
                  ('knn', KNeighborsClassifier())]).fit(Xk_tr, yk_tr)
withsc = Pipeline([('imp', SimpleImputer(strategy='median')),
                   ('sc', StandardScaler()),
                   ('knn', KNeighborsClassifier())]).fit(Xk_tr, yk_tr)

print(f'\nkNN without scaling: {accuracy_score(yk_te, plain.predict(Xk_te)):.4f}')
print(f'kNN with    scaling: {accuracy_score(yk_te, withsc.predict(Xk_te)):.4f}')

✅ **Expected:** `0.6034` → `0.6257`

`fare` runs 0–512 while `pclass` runs 1–3. Before scaling, the distance between two passengers
depends on almost nothing but the ticket price.

**A decision tree has no such problem**, because it asks one column at a time whether a value is above a
threshold, and thresholds do not care about units. That is why we opened the session with a tree.

### 🔵 C.6 — Imbalance: where accuracy lies to your face

Titanic is 62:38, which is not skewed enough to show the problem.
So we will **randomly drop survivors until they are 10% of the data, purely as a demonstration**
(this version is deliberately distorted — it is not real data, it is here to show the effect).

In [ ]:
from sklearn.metrics import f1_score

died = data[data.survived == 0]
kept = data[data.survived == 1].sample(n=int(len(died) * 0.10 / 0.90), random_state=SEED)
imb  = pd.concat([died, kept]).sample(frac=1, random_state=SEED)
print('class balance now:', imb.survived.value_counts(normalize=True).round(3).to_dict())

Xi, yi = imb.drop(columns=['survived'])[NUM], imb['survived']
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.2, random_state=SEED, stratify=yi)

dumb = DummyClassifier(strategy='most_frequent').fit(Xi_tr, yi_tr)
tree_i = Pipeline([('imp', SimpleImputer(strategy='median')),
                   ('tree', DecisionTreeClassifier(random_state=SEED))]).fit(Xi_tr, yi_tr)

for name, model in [('always guess "died"', dumb), ('decision tree', tree_i)]:
    pred = model.predict(Xi_te)
    print(f'{name:22s} accuracy {accuracy_score(yi_te, pred):.4f}   F1 (survivors) {f1_score(yi_te, pred):.4f}')

✅ **Expected:**

| Model | Accuracy | F1 (survivors) |
|---|---:|---:|
| always guess "died" | **0.9016** | 0.0000 |
| Decision Tree | 0.8443 | **0.2963** |

**Read that table carefully, because it inverts your instinct.**

By accuracy alone, guessing "died" every time **beats** the decision tree by 5.7 points.
But that winning model **never finds a single survivor** (F1 = 0). It learned nothing; it repeats one answer.

The decision tree, with the lower accuracy, is the only one doing the job we hired it for.

> **Pick a model by accuracy on skewed data and you will pick the worse model every time.**

### 🔵 C.7 — The kind of leakage you cannot see

The leakage in C.3 was loud: 100%. This one is silent.

If you `fit` an imputer or a scaler on the **whole** dataset before splitting into train and test,
the statistics it learns (median, mean, std) contain information from the test set —
so the model has already peeked at part of the exam.

In [ ]:
median_all   = X['age'].median()
median_train = Xk_tr['age'].median()

print(f'median age from ALL data  : {median_all}')
print(f'median age from TRAIN only: {median_train}')
print(f'mean fare  from ALL data  : {X["fare"].mean():.4f}')
print(f'mean fare  from TRAIN only: {Xk_tr["fare"].mean():.4f}')

✅ **Expected:** median `28.0` vs `28.5` · mean fare `32.2042` vs `31.8198`

**To be straight with you:** on Titanic the difference is tiny and accuracy barely moves.

It becomes serious when the dataset is **small**, or when the transform is stronger
(target encoding, SMOTE). And the real problem is that **you cannot see it happening** —
you find out when the model goes live and the numbers drop.

> **The answer is not to be careful. It is to use a tool that cannot make the mistake** — a `Pipeline`.

### 🔵 C.8 — `Pipeline`, the tool that makes C.7 impossible

A `Pipeline` ties every step together and fits the whole chain on **train only**, automatically.

`ColumnTransformer` sends numeric and text columns down separate routes, because they need different treatment.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

NUM = ['age', 'sibsp', 'parch', 'fare', 'pclass']
CAT = ['sex', 'embarked', 'embark_town', 'alone']

preprocess = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('scale',  StandardScaler())]), NUM),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                      ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CAT),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

full = Pipeline([('prep', preprocess),
                 ('model', DecisionTreeClassifier(random_state=SEED))]).fit(X_train, y_train)

print(f'Accuracy: {accuracy_score(y_test, full.predict(X_test)):.4f}')

✅ **Expected:** `Accuracy: 0.8101`

**`handle_unknown='ignore'` matters more than it looks.** If the test set contains a value never seen in
training — a port that only appears once, say — the default behaviour is to crash mid-run.
This option encodes it as all zeros instead.

### 🟠 Your-Turn C — Build the same pipeline for your group's data

Take the data from Your-Turn B and write a `Pipeline` that runs without errors.

It needs all four: **numeric and text columns separated** · **missing values imputed** ·
**text encoded** · **`train_test_split` before any `fit`, always**.

Before you write it, answer this: **does your data have an `alive` column of its own?**
Anything you could only know once you already know the answer.

✅ **Expected:** one accuracy number that runs — **with its baseline printed next to it.**

In [ ]:
# TODO: a pipeline for your group's data
# MY_NUM = [...]
# MY_CAT = [...]
# MY_TARGET = '...'

---
# D · Back to the Model — and Measuring It Properly
**about 35 minutes**

Part A opened a loop: the model broke because the data was dirty. This part closes it **with numbers**,
and then corrects the most dangerous misunderstanding of the session: accuracy is not the answer.

### 🔵 D.1 — Before and after

In [ ]:
rows = [
    ('Baseline (always "died")', base_acc),
    ('A: tree, 3 raw columns',   acc),
    ('C: tree, full pipeline',   accuracy_score(y_test, full.predict(X_test))),
]
before_after = pd.DataFrame(rows, columns=['Stage', 'Accuracy'])
before_after['vs baseline'] = (before_after['Accuracy'] - base_acc).round(4)
print(before_after.round(4).to_string(index=False))

✅ **Expected:** baseline `0.6145` → three raw columns `0.6480` → full pipeline `0.8101`

Read it two ways:
- **Against the baseline:** the full pipeline is **+19.6 points** better than guessing; the three raw columns
  were only **+3.4** better
- **What part C bought:** `0.8101 − 0.6480 = ` **+16.2 points** — that is the value of preprocessing alone,
  because the model is still the same decision tree

Keep the number **16.2** for part D.2.

### 🔵 D.2 — How much does changing the model help?

Same `preprocess`, only the final estimator changes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score

candidates = {
    'Baseline (majority)': DummyClassifier(strategy='most_frequent'),
    'kNN (k=5)':           KNeighborsClassifier(),
    'Decision Tree':       DecisionTreeClassifier(random_state=SEED),
    'Naive Bayes':         GaussianNB(),
    'Random Forest':       RandomForestClassifier(n_estimators=300, random_state=SEED),
    'Gradient Boosting':   GradientBoostingClassifier(random_state=SEED),
}

results = []
for name, model in candidates.items():
    pipe = Pipeline([('prep', preprocess), ('model', model)]).fit(X_train, y_train)
    pred = pipe.predict(X_test)
    auc  = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
    results.append((name, accuracy_score(y_test, pred), f1_score(y_test, pred), auc))

scores = pd.DataFrame(results, columns=['Model', 'Accuracy', 'F1', 'ROC-AUC'])
print(scores.round(4).to_string(index=False))

✅ **Expected:**

| Model | Accuracy | F1 | ROC-AUC |
|---|---:|---:|---:|
| Baseline (majority) | 0.6145 | 0.0000 | 0.5000 |
| kNN (k=5) | 0.8045 | 0.7328 | 0.8391 |
| Decision Tree | 0.8101 | 0.7463 | 0.7819 |
| Naive Bayes | 0.7765 | 0.7101 | 0.8022 |
| Random Forest | 0.8101 | 0.7344 | 0.8245 |
| Gradient Boosting | 0.7989 | 0.7097 | 0.8211 |

**The most important observation in the session:** the five real models span **0.7765–0.8101** —
**3.4 points** apart. The preprocessing in part C was worth **16.2 points**, nearly **five times** as much.

> Time spent understanding your data pays better than time spent hunting for a fancier model.

**And one more thing to notice:** `Decision Tree` has the highest accuracy (0.8101) but the
**lowest ROC-AUC of the real models** (0.7819), while `kNN` has slightly lower accuracy and the
**highest ROC-AUC** (0.8391).

They measure different things. Accuracy asks how many predictions are right at a 0.5 threshold.
ROC-AUC asks how well the model **ranks** risk, across every threshold.
If the job is "send the 20 highest-risk people for further screening", ROC-AUC is the relevant one.

### 🔵 D.3 — Overfitting, as a picture

`max_depth` is the complexity dial on a tree. Sweep it and plot train and test scores together.

In [ ]:
depths = [1, 2, 3, 4, 5, 7, 10, 15, 20]
train_scores, test_scores = [], []

for d in depths:
    p = Pipeline([('prep', preprocess),
                  ('model', DecisionTreeClassifier(max_depth=d, random_state=SEED))]).fit(X_train, y_train)
    train_scores.append(p.score(X_train, y_train))
    test_scores.append(p.score(X_test, y_test))

plt.figure(figsize=(7, 4))
plt.plot(depths, train_scores, 'o-', label='train', color='tab:blue')
plt.plot(depths, test_scores, 's-', label='test', color='tab:orange')
plt.fill_between(depths, test_scores, train_scores, color='tab:red', alpha=0.08)
plt.text(10.4, 0.885, 'the gap is what the model memorised', color='tab:red', fontsize=9)
plt.xlabel('max_depth'); plt.ylabel('accuracy')
plt.title('The tree keeps learning the training set - and stops helping on new data')
plt.legend(); plt.ylim(0.6, 1.02); plt.tight_layout(); plt.show()

print(pd.DataFrame({'max_depth': depths, 'train': np.round(train_scores, 4),
                    'test': np.round(test_scores, 4)}).to_string(index=False))

✅ **Expected:** train climbs from `0.7893` to `0.9817`, while test wanders around `0.76–0.82` and goes nowhere

**This is what overfitting looks like** — the red area between the lines is what the model memorised
and cannot reuse on anyone new.

This chart is the **bias–variance tradeoff** your lectures cover, drawn from real numbers:

| End of the curve | What is happening | The name for it |
|---|---|---|
| `max_depth=1`, both scores low and close | the model is too simple to capture the pattern — it is wrong in the same way every time | **high bias** (underfitting) |
| `max_depth=20`, train 0.98 and test 0.81 | the model has memorised this particular sample — a different sample would give a different model | **high variance** (overfitting) |

The useful depth is where the two lines are closest without both being low, and the red area is the
variance you are paying for.

**Notice something else:** the test line has no clear peak. `max_depth=15` gives `0.8212`,
but `max_depth=10` gives `0.7989`, which is worse than `max_depth=7`.
That wobble is not a signal from the data — it is **noise from splitting the test set once.**
Picking `max_depth` straight off this chart means picking whichever value got lucky,
which is exactly why the next section exists.

### 🔵 D.4 — One split is not enough

Every number so far comes from **one** train/test split, with `random_state=42`.
Change that number and the results change. So which one do you report?

**Cross-validation** splits several times and reports both the average and the spread.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rf = Pipeline([('prep', preprocess),
               ('model', RandomForestClassifier(n_estimators=300, random_state=SEED))])

folds = cross_val_score(rf, X, y, cv=cv, scoring='f1')
print('F1 per fold:', np.round(folds, 4))
print(f'mean {folds.mean():.4f}  ±{folds.std():.4f}')

✅ **Expected:** `[0.7852 0.7465 0.7015 0.7669 0.7669]` · mean `0.7534` ±`0.0287`

Lowest and highest fold are **8 points** apart, on the same data with the same model.

> **How to report a result:** write `0.75 ± 0.03`, not `0.7852`.
> Reporting a single number from a single split is reporting your luckiest value.

### 🔵 D.5 — How is the model wrong?

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

rf_fitted = rf.fit(X_train, y_train)
pred = rf_fitted.predict(X_test)

cm = confusion_matrix(y_test, pred)
print(cm)
print()
print(classification_report(y_test, pred, target_names=['died', 'survived'], digits=3))

✅ **Expected:** the matrix `[[98 12] [22 47]]`, and recall for `survived` = `0.681`

**Reading the matrix:**

| | predicted: died | predicted: survived |
|---|---:|---:|
| **actually died** | 98 ✅ | 12 ❌ *false positive* |
| **actually survived** | **22 ❌ *false negative*** | 47 ✅ |

The model **misses 22 of the 69 survivors** — recall of 68%, while accuracy reads 81%.
The accuracy figure never mentioned this.

### 🔵 D.6 — The question to answer before choosing a metric

Suppose this were **screening patients for heart-attack risk**, where 1 = at risk.

- **False negative** = telling a sick patient they are fine → they go home untreated
- **False positive** = telling a healthy patient they are at risk → extra tests, cost and worry, but they are safe

The two errors cost wildly different amounts. **Screening therefore optimises recall**, and accepts
a lot of false positives to get it.

> **The order is: understand the problem → choose the metric → then tune the model.**
> Not: pick whichever metric makes the number look best.

### 🎨 D.7 — Chart polish, step 1

Every session ends with ten minutes spent on a chart you just made yourself. **Today's two techniques:**

1. **A chart title should be the conclusion, not the column name.** `"Model accuracy"` tells the reader nothing.
   `"Every model lands within 3 points of the others"` tells them the finding.
2. **A y-axis that does not start at zero magnifies small differences** — the most common way to lie with a chart.

In [ ]:
plot_df = scores[scores.Model != 'Baseline (majority)']

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))

# left: the misleading version
ax[0].bar(range(len(plot_df)), plot_df.Accuracy, color='tab:blue')
ax[0].set_ylim(0.77, 0.815)
ax[0].set_xticks(range(len(plot_df)))
ax[0].set_xticklabels(plot_df.Model, rotation=20, ha='right', fontsize=8)
ax[0].set_title('Model accuracy')                       # column name as a title
ax[0].set_ylabel('accuracy')

# right: honest axis + a title that states the finding
colors = ['tab:grey'] * len(plot_df)
colors[int(np.argmax(plot_df.Accuracy.values))] = 'tab:blue'
ax[1].bar(range(len(plot_df)), plot_df.Accuracy, color=colors)
ax[1].axhline(base_acc, color='tab:red', ls='--', lw=1)
ax[1].text(-0.4, base_acc + 0.012, f'baseline {base_acc:.2f}', color='tab:red', fontsize=9)
ax[1].set_ylim(0, 1)
ax[1].set_xticks(range(len(plot_df)))
ax[1].set_xticklabels(plot_df.Model, rotation=20, ha='right', fontsize=8)
ax[1].set_title('Every model lands within 3 points - the data mattered more than the model')
ax[1].set_ylabel('accuracy')

plt.tight_layout(); plt.show()

✅ **Expected:** two panels built from **the same numbers** that tell different stories

The left one makes the Decision Tree look like a clear winner. The right one shows every model bunched
together, with a baseline to measure against.

**The right one is the honest chart:** its axis starts at zero, everything is grey except the bar being
pointed at, and the title states the finding instead of naming the chart.

### 🟠 Your-Turn D — Which metric does your group's data need?

Write **three or four lines** in the markdown cell below, covering all three of:

1. Is your data balanced? (give the numbers)
2. **Which costs more in your problem, a false negative or a false positive** — and why
3. Your one primary metric, with a reason that follows from point 2

✅ **Expected:** an answer tied to your actual problem, not *"accuracy, because it is easy to understand"*

In [ ]:
# (no code required - use this cell to check the class balance in your group's data)
# TODO:

**Your group's answer (write here):**

1. Class balance:
2. False negative vs false positive:
3. Primary metric, and why:

---
# E · When the Answer Is a Number, Not a Label
**about 20 minutes**

Everything so far predicted a label: survived or not. Plenty of real questions want a **quantity** —
what will this cost, how many will arrive, how much will burn. Four of the twenty assignment topics
are exactly that.

The pipeline does not change. **The way you score it changes completely.**

### 🔵 E.1 — Same pipeline, different question

California housing: 20,640 districts, and the target is the median house value in that district.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor

homes = fetch_california_housing(as_frame=True)
Xh, yh = homes.data, homes.target
print('shape:', Xh.shape, '| target: median house value in $100,000s')

Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, test_size=0.2, random_state=SEED)

def numeric_pipeline(model):
    return Pipeline([('impute', SimpleImputer(strategy='median')),
                     ('scale',  StandardScaler()),
                     ('model',  model)])

tree_reg = numeric_pipeline(DecisionTreeRegressor(random_state=SEED)).fit(Xh_tr, yh_tr)
print('trained')

### 🔵 E.2 — The baseline still comes first

For a label, the dumbest model answers with the most common class. For a number, it answers with the
**median every time** — and the three regression metrics all disagree about how bad that is.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

candidates_reg = {
    'Baseline (always the median)': DummyRegressor(strategy='median'),
    'Decision Tree':                DecisionTreeRegressor(random_state=SEED),
    'Random Forest':                RandomForestRegressor(n_estimators=100, random_state=SEED),
}

reg_rows = []
for name, model in candidates_reg.items():
    fitted = numeric_pipeline(model).fit(Xh_tr, yh_tr)
    pred = fitted.predict(Xh_te)
    reg_rows.append((name,
                     mean_absolute_error(yh_te, pred),
                     np.sqrt(mean_squared_error(yh_te, pred)),
                     r2_score(yh_te, pred)))

print(pd.DataFrame(reg_rows, columns=['Model', 'MAE', 'RMSE', 'R2']).round(4).to_string(index=False))

✅ **Expected:**

| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
| Baseline (always the median) | 0.8740 | 1.1731 | **-0.0502** |
| Decision Tree | 0.4548 | 0.7060 | 0.6196 |
| Random Forest | 0.3276 | 0.5055 | 0.8050 |

**Three things to read off that table:**

1. **R² can be negative.** The baseline scores -0.05, which means it is worse than predicting the *mean*.
   R² is not a percentage and it does not stop at zero.
2. **RMSE is always larger than MAE**, and the gap tells you something: RMSE squares each error before
   averaging, so a few large misses move it far more than they move MAE. Random Forest is 0.33 by MAE
   and 0.51 by RMSE — that distance is where the big mistakes live.
3. Pick which one matters **before** you train. If being wrong by \$200,000 once is worse than being
   wrong by \$20,000 ten times, you care about RMSE. If every error costs the same per dollar, MAE.

### 🔵 E.3 — Where the errors actually are

Topic 2 in the assignment bank warns that this target is capped at 5.0. Here is what that costs you.

In [ ]:
best = numeric_pipeline(RandomForestRegressor(n_estimators=100, random_state=SEED)).fit(Xh_tr, yh_tr)
pred_h = best.predict(Xh_te)

at_ceiling = yh_te == yh_te.max()
print(f'districts at the 5.0 ceiling : {at_ceiling.sum()} of {len(yh_te)} ({at_ceiling.mean():.1%})')
print(f'MAE on those districts       : {mean_absolute_error(yh_te[at_ceiling], pred_h[at_ceiling]):.4f}')
print(f'MAE on everything else       : {mean_absolute_error(yh_te[~at_ceiling], pred_h[~at_ceiling]):.4f}')

residual = yh_te - pred_h
plt.figure(figsize=(7, 4))
plt.scatter(pred_h, residual, s=6, alpha=0.25, color='tab:blue')
plt.scatter(pred_h[at_ceiling], residual[at_ceiling], s=8, alpha=0.6, color='tab:red')
plt.axhline(0, color='grey', lw=1)
plt.text(1.2, 2.2, 'red = districts capped at 5.0', color='tab:red', fontsize=9)
plt.xlabel('predicted value'); plt.ylabel('actual - predicted')
plt.title('The model cannot reach the districts the census clipped')
plt.tight_layout(); plt.show()

✅ **Expected:** 179 of 4,128 districts (4.3%) · MAE on them `0.7339` against `0.3091` everywhere else —
**more than twice as bad**

The red band running along the top is not a modelling failure you can tune away. Those districts were
worth more than 5.0 and the census wrote down 5.0, so the information is gone before you start.

> **Lecture 04 calls this a residual plot.** Residuals scattered evenly around zero mean the model has
> nothing systematic left to learn. A band, a curve, or a fan means it does — and here the band tells you
> the problem is in the data, not the model.

### 🟠 Your-Turn E — Which metric fits your group's topic?

If your group picked topic 2, 3, 10 or 5, you are doing regression. Everyone else: answer it anyway for
the regression version of your question.

Write **3–4 lines**: does one large error hurt more than several small ones in your problem, which of
MAE and RMSE follows from that, and what a sensible baseline would be for your target.

✅ **Expected:** a metric chosen from how the errors cost, not from which number looked better

In [ ]:
# TODO: baseline + MAE/RMSE/R2 on your own target (or on the housing data again)

---
# Session 1 Wrap-Up

### Three things to remember

1. **A number without a baseline means nothing.** 90% accuracy can mean the model learned nothing at all (C.6)
2. **A result that is too good is a warning, not a win.** The 100% in C.3 was target leakage
3. **Time spent on the data beats time spent on the model.** Preprocessing bought 16.2 points; switching models bought 3.4
4. **A baseline exists for every kind of answer.** Most-common-class for a label, the median for a number — and R² below zero means you lost to it

### What to hand in
- This notebook, running end to end, with all four Your-Turn sections done
- The **before/after** preprocessing table, with the baseline beside it
- A `Pipeline` that runs on both titanic and your group's own data
- Your group's chosen topic from the Assignment 1 list in part B, with `TOPIC_ID` set

### Next session
All session we had the answer key in hand — the `survived` column. **What happens when there isn't one?**